# Stage 1 Results — Symbol Detection Analysis
This notebook loads your saved training metrics and best model checkpoint
and walks through every result in plain English.

**Sections:**
1. Setup & load results
2. Did training go well? (loss curves)
3. How accurate is the detector? (mAP, precision, recall)
4. Which symbol classes does it struggle with? (per-class breakdown)
5. What does it actually detect? (visual examples)
6. Plain-English summary & next steps

## 1. Setup

In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import display, Image as IPImage

import torch
from torch.utils.data import DataLoader

# ── adjust these paths if needed ──────────────────────────────────────────
PROJECT_ROOT  = Path("..")                          # notebook is in notebooks/
METRICS_PATH  = PROJECT_ROOT / "outputs/stage1/results/metrics.json"
TEST_METRICS  = PROJECT_ROOT / "outputs/stage1/results/test_metrics.json"
CHECKPOINT    = PROJECT_ROOT / "outputs/stage1/checkpoints/best.pt"
MUSCIMA_DIR   = PROJECT_ROOT / "data/raw/muscima"
PLOTS_DIR     = PROJECT_ROOT / "outputs/stage1/results/plots"
# ──────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F9FAFB',
    'axes.grid': True, 'grid.color': '#E5E7EB',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'lines.linewidth': 2.2,
})
COLORS = {'blue':'#2563EB','green':'#16A34A','red':'#DC2626',
          'amber':'#D97706','purple':'#7C3AED','gray':'#6B7280'}

print('Setup complete.')

In [ ]:
# Load training history
with open(METRICS_PATH) as f:
    H = json.load(f)

# Load test metrics if available
test_metrics = None
if TEST_METRICS.exists():
    with open(TEST_METRICS) as f:
        test_metrics = json.load(f)

epochs      = H['epoch']
num_epochs  = len(epochs)
best_epoch  = epochs[int(np.argmax(H['val_map']))]
best_map    = max(H['val_map'])

print(f'Epochs trained   : {num_epochs}')
print(f'Best val mAP     : {best_map:.4f}  (epoch {best_epoch})')
print(f'Final train loss : {H["train_loss_total"][-1]:.4f}')
print(f'Final val mAP    : {H["val_map"][-1]:.4f}')

## 2. Did Training Go Well?

**What to look for:**
- Loss should trend **downward** over epochs — a flat or increasing loss means the model isn't learning
- The four component losses (classifier, box regression, objectness, RPN) should all decrease
- If loss drops fast then plateaus, training converged — more epochs won't help much
- If loss is still falling at the last epoch, consider training longer

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Loss Curves', fontsize=15, fontweight='bold')

# Total loss
ax1.plot(epochs, H['train_loss_total'], color=COLORS['blue'], marker='o', label='Total Loss')
ax1.set_title('Total Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Add a trend annotation
first, last = H['train_loss_total'][0], H['train_loss_total'][-1]
pct = (first - last) / first * 100
ax1.text(0.05, 0.05, f'↓ {pct:.1f}% reduction', transform=ax1.transAxes,
         fontsize=10, color=COLORS['green'],
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Component losses
components = [
    ('train_loss_classifier',  'Classifier',  COLORS['blue']),
    ('train_loss_box_reg',     'Box Reg',     COLORS['green']),
    ('train_loss_objectness',  'Objectness',  COLORS['red']),
    ('train_loss_rpn_box_reg', 'RPN Box Reg', COLORS['purple']),
]
for key, label, color in components:
    if key in H and any(v > 0 for v in H[key]):
        ax2.plot(epochs, H[key], color=color, marker='o', label=label)
ax2.set_title('Loss Component Breakdown')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

# Plain-English diagnosis
print('\n--- DIAGNOSIS ---')
if pct > 50:
    print(f'✅  Loss dropped {pct:.1f}% — training converged well.')
elif pct > 20:
    print(f'⚠️   Loss dropped {pct:.1f}% — some learning happened but may need more epochs.')
else:
    print(f'❌  Loss only dropped {pct:.1f}% — model may not have learned much. Check your data and learning rate.')

if H['train_loss_total'][-1] < H['train_loss_total'][-2]:
    print('📈  Loss was still decreasing at the final epoch — consider training longer.')
else:
    print('✅  Loss had levelled off by the final epoch — good stopping point.')

## 3. How Accurate Is the Detector?

**What these numbers mean in plain English:**

| Metric | What it measures | Good score |
|--------|-----------------|------------|
| **mAP** | Average accuracy across all symbol classes at IoU=0.5 | > 0.5 is solid for OMR |
| **Precision** | Of all boxes the model drew, how many were correct? | > 0.6 |
| **Recall** | Of all real symbols, how many did the model find? | > 0.6 |

A high precision + low recall = model is picky, misses a lot of symbols  
A low precision + high recall = model finds everything but adds many false detections  
Both high = good detector

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Metrics Over Training', fontsize=15, fontweight='bold')

# Precision & Recall
ax1.plot(epochs, H['val_precision'], color=COLORS['blue'],  marker='o', label='Precision')
ax1.plot(epochs, H['val_recall'],    color=COLORS['green'], marker='s', label='Recall')
ax1.axhline(0.5, color=COLORS['gray'], linestyle='--', linewidth=1, alpha=0.7, label='0.5 threshold')
ax1.set_ylim(0, 1.05)
ax1.set_title('Precision & Recall')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Score')
ax1.legend()

# mAP
ax2.plot(epochs, H['val_map'], color=COLORS['red'], marker='o', label='mAP @ IoU=0.5')
best_idx = int(np.argmax(H['val_map']))
ax2.scatter([epochs[best_idx]], [H['val_map'][best_idx]],
            color=COLORS['red'], s=150, zorder=5)
ax2.annotate(f"Best: {H['val_map'][best_idx]:.3f} (epoch {epochs[best_idx]})",
             xy=(epochs[best_idx], H['val_map'][best_idx]),
             xytext=(15, -20), textcoords='offset points',
             fontsize=10, color=COLORS['red'],
             arrowprops=dict(arrowstyle='->', color=COLORS['red']))
ax2.set_ylim(0, 1.05)
ax2.set_title('Mean Average Precision (mAP)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mAP')
ax2.legend()

plt.tight_layout()
plt.show()

# Summary table
final_p = H['val_precision'][-1]
final_r = H['val_recall'][-1]
final_m = H['val_map'][-1]

print('\n--- FINAL EPOCH SUMMARY ---')
print(f'  Precision : {final_p:.4f}  {"✅" if final_p > 0.5 else "⚠️"}')
print(f'  Recall    : {final_r:.4f}  {"✅" if final_r > 0.5 else "⚠️"}')
print(f'  mAP       : {final_m:.4f}  {"✅" if final_m > 0.4 else "⚠️"}')

if final_p > final_r + 0.15:
    print('\n💡  Precision is much higher than recall — the model is missing symbols.')
    print('    Try lowering the confidence threshold in SymbolDetector (currently 0.5).')
elif final_r > final_p + 0.15:
    print('\n💡  Recall is much higher than precision — the model has many false detections.')
    print('    Try raising the confidence threshold, or train for more epochs.')
else:
    print('\n✅  Precision and recall are reasonably balanced.')

## 4. Which Symbol Classes Does It Struggle With?

Not all music symbols are equally easy to detect. Small, dense symbols like
stems and ledger lines are much harder than large, distinctive ones like clefs.
This chart shows per-class AP — how well the model detects each individual symbol type.

In [ ]:
if test_metrics and 'per_class' in test_metrics:
    pc = test_metrics['per_class']

    classes = list(pc.keys())
    aps     = [pc[c]['ap']        for c in classes]
    precs   = [pc[c]['precision'] for c in classes]
    recs    = [pc[c]['recall']    for c in classes]

    # Sort by AP descending
    order   = np.argsort(aps)[::-1]
    classes = [classes[i] for i in order]
    aps     = [aps[i]     for i in order]
    precs   = [precs[i]   for i in order]
    recs    = [recs[i]    for i in order]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(6, len(classes)*0.45)))
    fig.suptitle('Per-Class Detection Performance (Test Set)', fontsize=14, fontweight='bold')

    # AP bar chart
    bar_colors = [COLORS['green'] if a >= 0.5 else
                  COLORS['amber'] if a >= 0.25 else
                  COLORS['red']   for a in aps]
    bars = ax1.barh(classes, aps, color=bar_colors, edgecolor='white', height=0.7)
    ax1.set_xlim(0, 1.05)
    ax1.axvline(0.5, color=COLORS['gray'], linestyle='--', linewidth=1)
    ax1.set_title('Average Precision per Class')
    ax1.set_xlabel('AP @ IoU=0.5')
    for bar, ap in zip(bars, aps):
        ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{ap:.3f}', va='center', fontsize=8)

    # Precision vs Recall scatter per class
    scatter = ax2.scatter(recs, precs, c=aps, cmap='RdYlGn',
                          vmin=0, vmax=1, s=100, edgecolors='gray', linewidths=0.5)
    for cls, p, r in zip(classes, precs, recs):
        ax2.annotate(cls, (r, p), fontsize=7, alpha=0.8,
                     xytext=(3, 3), textcoords='offset points')
    plt.colorbar(scatter, ax=ax2, label='AP')
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_ylim(-0.05, 1.05)
    ax2.axhline(0.5, color=COLORS['gray'], linestyle='--', linewidth=1, alpha=0.5)
    ax2.axvline(0.5, color=COLORS['gray'], linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_title('Precision vs Recall per Class')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')

    plt.tight_layout()
    plt.show()

    # Plain-English class breakdown
    print('\n--- CLASS BREAKDOWN ---')
    strong = [(c, a) for c, a in zip(classes, aps) if a >= 0.5]
    weak   = [(c, a) for c, a in zip(classes, aps) if a < 0.25]

    if strong:
        print(f'\n✅  Strong classes (AP ≥ 0.5): {len(strong)}')
        for c, a in strong:
            print(f'    {c:<25s}  AP={a:.3f}')

    if weak:
        print(f'\n❌  Weak classes (AP < 0.25): {len(weak)}')
        for c, a in weak:
            print(f'    {c:<25s}  AP={a:.3f}')
        print('\n💡  Weak classes are often small, dense symbols (stems, beams, ledger lines).')
        print('    Consider: more training epochs, data augmentation, or higher input resolution.')
else:
    print('Per-class test metrics not found.')
    print('Run evaluate_and_plot() from train.py to generate them:')
    print('')
    print('  from src.stage1_detection.train import evaluate_and_plot')
    print('  evaluate_and_plot("outputs/stage1/checkpoints/best.pt", "data/raw/muscima")')

## 5. What Does It Actually Detect?

Run the model on individual test images and see ground truth vs predictions side by side.
Green boxes = ground truth, colored boxes = predictions.
Each predicted box shows the class label and confidence score (0–1).

In [ ]:
# Show any already-saved detection images from training
saved_plots = sorted(PLOTS_DIR.glob('detection_sample_*.png'))

if saved_plots:
    print(f'Found {len(saved_plots)} saved detection images from training.\n')
    for p in saved_plots[:3]:   # show first 3
        print(f'  {p.name}')
        display(IPImage(str(p), width=900))
else:
    print('No saved detection images found. Generating now...')
    print('(This requires the model checkpoint and dataset to be available.)')

In [ ]:
# ── Live inference on a single image ─────────────────────────────────────
# Requires: checkpoint + dataset. Comment out if not available.

from src.stage1_detection.dataset import MUSCIMADataset, IDX_TO_CLASS
from src.stage1_detection.detector import build_faster_rcnn

CONFIDENCE_THRESHOLD = 0.5
SAMPLE_IDX           = 0      # change to see different images

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = build_faster_rcnn(pretrained_backbone=False)
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  '
          f'(val mAP={ckpt["val_map"]:.4f})')
else:
    print('⚠️  Checkpoint not found — using random weights.')

model.to(device)
model.eval()

# Load dataset
test_ds = MUSCIMADataset(str(MUSCIMA_DIR), split='test')
image_tensor, target = test_ds[SAMPLE_IDX]

with torch.no_grad():
    output = model([image_tensor.to(device)])[0]

image_np    = image_tensor.permute(1,2,0).numpy()
pred_boxes  = output['boxes'].cpu().tolist()
pred_scores = output['scores'].cpu().tolist()
pred_labels = output['labels'].cpu().tolist()
gt_boxes    = target['boxes'].tolist()
gt_labels   = target['labels'].tolist()

BOX_COLORS  = ['#EF4444','#3B82F6','#10B981','#F59E0B','#8B5CF6',
               '#EC4899','#06B6D4','#84CC16','#F97316','#6366F1']

fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle(f'Test Sample {SAMPLE_IDX}', fontsize=13, fontweight='bold')

for ax, title in [(ax_gt, 'Ground Truth'), (ax_pred, 'Model Predictions')]:
    ax.imshow(image_np)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

# Ground truth
for box, label in zip(gt_boxes, gt_labels):
    x1,y1,x2,y2 = box
    color = BOX_COLORS[label % len(BOX_COLORS)]
    ax_gt.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                    linewidth=1.5, edgecolor=color, facecolor='none'))
    ax_gt.text(x1, max(0,y1-3), IDX_TO_CLASS.get(label,'?'),
               fontsize=6, color=color,
               bbox=dict(facecolor='white',alpha=0.6,pad=1,edgecolor='none'))

# Predictions
high_conf = [(b,s,l) for b,s,l in zip(pred_boxes,pred_scores,pred_labels)
             if s >= CONFIDENCE_THRESHOLD]
ax_pred.set_title(f'Predictions  ({len(high_conf)} symbols ≥ {CONFIDENCE_THRESHOLD:.0%} confidence)',
                  fontsize=11)
for box, score, label in high_conf:
    x1,y1,x2,y2 = box
    color = BOX_COLORS[label % len(BOX_COLORS)]
    ax_pred.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                      linewidth=1.5, edgecolor=color, facecolor='none'))
    ax_pred.text(x1, max(0,y1-3), f'{IDX_TO_CLASS.get(label,"?")}: {score:.2f}',
                 fontsize=6, color=color,
                 bbox=dict(facecolor='white',alpha=0.6,pad=1,edgecolor='none'))

plt.tight_layout()
plt.show()

print(f'\nGround truth symbols : {len(gt_boxes)}')
print(f'Predicted symbols    : {len(high_conf)}  (at ≥{CONFIDENCE_THRESHOLD:.0%} confidence)')
print(f'All predictions made : {len(pred_boxes)}  (including low-confidence)')

In [ ]:
# ── Confidence score distribution ────────────────────────────────────────
# Shows whether the model is confident or uncertain about its predictions.
# A healthy detector should have a clear split: many high-confidence true
# positives and many low-confidence false positives.

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pred_scores, bins=20, range=(0,1),
        color=COLORS['blue'], edgecolor='white', alpha=0.8)
ax.axvline(CONFIDENCE_THRESHOLD, color=COLORS['red'],
           linestyle='--', linewidth=2,
           label=f'Threshold = {CONFIDENCE_THRESHOLD}')
ax.set_title('Prediction Confidence Score Distribution (this image)', fontweight='bold')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Number of Predictions')
ax.legend()
plt.tight_layout()
plt.show()

above = sum(1 for s in pred_scores if s >= CONFIDENCE_THRESHOLD)
below = len(pred_scores) - above
print(f'Predictions above threshold : {above}')
print(f'Predictions below threshold : {below}  (filtered out)')
if below > above * 3:
    print('💡  Many low-confidence predictions — the model is uncertain on this image.')
    print('    This is normal for complex or dense sheet music pages.')

## 6. Plain-English Summary & Next Steps

In [ ]:
print('=' * 55)
print('  STAGE 1 RESULTS SUMMARY')
print('=' * 55)

final_map  = H['val_map'][-1]
final_prec = H['val_precision'][-1]
final_rec  = H['val_recall'][-1]
loss_drop  = (H['train_loss_total'][0] - H['train_loss_total'][-1]) / H['train_loss_total'][0] * 100

print(f'\n  Epochs trained    : {num_epochs}')
print(f'  Loss reduction    : {loss_drop:.1f}%')
print(f'  Best val mAP      : {best_map:.4f}  (epoch {best_epoch})')
print(f'  Final precision   : {final_prec:.4f}')
print(f'  Final recall      : {final_rec:.4f}')

if test_metrics:
    print(f'\n  Test set mAP      : {test_metrics["map"]:.4f}')
    print(f'  Test precision    : {test_metrics["precision"]:.4f}')
    print(f'  Test recall       : {test_metrics["recall"]:.4f}')

print('\n' + '=' * 55)
print('  WHAT THIS MEANS FOR YOUR PROJECT')
print('=' * 55)

if best_map >= 0.5:
    readiness = 'READY'
    msg = 'Stage 1 is performing well. The detected symbols should provide\n  good input to Stage 2.'
elif best_map >= 0.3:
    readiness = 'USABLE'
    msg = 'Stage 1 is producing reasonable detections. Stage 2 will need\n  to be robust to some detection errors.'
else:
    readiness = 'NEEDS WORK'
    msg = 'Stage 1 accuracy is low. Stage 2 results will be noisy.\n  Consider more training or data augmentation.'

print(f'\n  Stage 1 status: {readiness}')
print(f'  {msg}')

print('\n' + '=' * 55)
print('  RECOMMENDED NEXT STEPS')
print('=' * 55)

steps = []
if H['train_loss_total'][-1] < H['train_loss_total'][-2]:
    steps.append('1. Train for more epochs — loss was still decreasing')
if final_prec > final_rec + 0.15:
    steps.append('2. Lower confidence threshold (try 0.3) to improve recall')
if final_rec > final_prec + 0.15:
    steps.append('2. Raise confidence threshold (try 0.7) to reduce false positives')
steps.append(f'{len(steps)+1}. Run stage1_analysis cell 5 on more test images to spot failure modes')
steps.append(f'{len(steps)+1}. Proceed to Stage 2 training once satisfied with detection quality')

print()
for s in steps:
    print(f'  {s}')
print()